# Fuzzy & Semantic Match Strategies

Interactive notebook demonstrating the two new string-matching strategies:

| Strategy | Config name | Dependency | Use case |
|---|---|---|---|
| **FuzzyMatchStrategy** | `fuzzy_match` | `difflib` (stdlib) | Partial string matches — "JPMorgan Chase" vs "JPMorgan" |
| **SemanticMatchStrategy** | `semantic_match` | `sentence-transformers` (optional) | Meaning-based matches — "Amount Paid" vs "Money Received" |

---

## Part 1 — Fuzzy Match (no extra dependencies)

In [1]:
from nerds_nlp.models.edge.matching.strategies import FuzzyMatchStrategy

print("FuzzyMatchStrategy imported (uses stdlib difflib only)")

FuzzyMatchStrategy imported (uses stdlib difflib only)


### 1.1 — Basic Pair Scoring

Compare individual string pairs to see the similarity ratio.

In [2]:
fuzzy = FuzzyMatchStrategy()

pairs = [
    ("JPMorgan Chase", "JPMorgan Chase"),       # identical
    ("JPMorgan Chase", "JPMorgan"),              # substring
    ("JPMorgan Chase", "JP Morgan Chase"),       # space variation
    ("ABC Corporation", "ABC Corp"),             # abbreviation
    ("Goldman Sachs", "Goldman Sachs Group"),    # extended name
    ("Deutsche Bank AG", "Deutsche Bank"),       # suffix dropped
    ("JPMorgan Chase", "Goldman Sachs"),         # completely different
    ("INCOMING", "incoming"),                    # case difference only
]

print(f"{'Input':<25s}  {'Candidate':<25s}  {'Score':>6s}")
print(f"{'-'*25}  {'-'*25}  {'-'*6}")
for a, b in pairs:
    score = fuzzy.score_pair(a, b)
    print(f"{a:<25s}  {b:<25s}  {score:>6.4f}")

Input                      Candidate                   Score
-------------------------  -------------------------  ------
JPMorgan Chase             JPMorgan Chase             1.0000
JPMorgan Chase             JPMorgan                   0.7273
JPMorgan Chase             JP Morgan Chase            0.9655
ABC Corporation            ABC Corp                   0.6957
Goldman Sachs              Goldman Sachs Group        0.8125
Deutsche Bank AG           Deutsche Bank              0.8966
JPMorgan Chase             Goldman Sachs              0.5185
INCOMING                   incoming                   1.0000


### 1.2 — Threshold Filtering

The `threshold` parameter forces low-similarity scores to 0.0.

In [3]:
thresholds = [0.0, 0.4, 0.6, 0.8]
test_pairs = [
    ("JPMorgan Chase", "JPMorgan"),
    ("ABC Corporation", "ABC Corp"),
    ("JPMorgan Chase", "Goldman Sachs"),
]

print(f"{'Input':<20s}  {'Candidate':<20s}", end="")
for t in thresholds:
    print(f"  {'t=' + str(t):>8s}", end="")
print()
print(f"{'-'*20}  {'-'*20}", end="")
for _ in thresholds:
    print(f"  {'-'*8}", end="")
print()

for a, b in test_pairs:
    print(f"{a:<20s}  {b:<20s}", end="")
    for t in thresholds:
        s = FuzzyMatchStrategy(threshold=t)
        score = s.score_pair(a, b)
        print(f"  {score:>8.4f}", end="")
    print()

Input                 Candidate                t=0.0     t=0.4     t=0.6     t=0.8
--------------------  --------------------  --------  --------  --------  --------
JPMorgan Chase        JPMorgan                0.7273    0.7273    0.7273    0.0000
ABC Corporation       ABC Corp                0.6957    0.6957    0.6957    0.0000
JPMorgan Chase        Goldman Sachs           0.5185    0.5185    0.0000    0.0000


### 1.3 — One-to-Many Scoring (score_many)

Score one input entity name against a pool of candidate entities.

In [4]:
fuzzy = FuzzyMatchStrategy()

input_name = "JPMorgan Chase & Co."
candidates = [
    "JPMorgan Chase & Co.",
    "JPMorgan Chase",
    "JP Morgan",
    "JPMorgan",
    "Morgan Stanley",
    "Goldman Sachs",
    "Bank of America",
    "Chase Bank",
]

scores = fuzzy.score_many(input_name, candidates)

# Sort by score descending
ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)

print(f"Input: '{input_name}'\n")
print(f"{'Rank':>4s}  {'Candidate':<30s}  {'Score':>6s}")
print(f"{'-'*4}  {'-'*30}  {'-'*6}")
for rank, (name, score) in enumerate(ranked, 1):
    print(f"{rank:>4d}  {name:<30s}  {score:>6.4f}")

Input: 'JPMorgan Chase & Co.'

Rank  Candidate                        Score
----  ------------------------------  ------
   1  JPMorgan Chase & Co.            1.0000
   2  JPMorgan Chase                  0.8235
   3  JPMorgan                        0.5714
   4  JP Morgan                       0.5517
   5  Morgan Stanley                  0.5294
   6  Goldman Sachs                   0.4242
   7  Chase Bank                      0.4000
   8  Bank of America                 0.2857


### 1.4 — End-to-End Fuzzy Matching via KKEdgeMatchingModel

Use `fuzzy_match` as a strategy in the full matching pipeline.

In [5]:
from nerds_nlp.models.edge.matching import KKEdgeMatchingModel

config_fuzzy = {
    "matching": {
        "threshold": 0.60,
        "fields": [
            {
                "name": "entity_name",
                "weight": 0.5,
                "strategy": "fuzzy_match",
                "params": {"threshold": 0.4},
            },
            {
                "name": "direction",
                "weight": 0.3,
                "strategy": "direction_inverse",
            },
            {
                "name": "settlement_date",
                "weight": 0.2,
                "strategy": "exact_match",
            },
        ],
    }
}

contract_fuzzy = {
    "input_documents": [
        {
            "id": "input-1",
            "links": [
                {
                    "entity_name": "JPMorgan Chase & Co.",
                    "direction": "incoming",
                    "settlement_date": "2025-06-15",
                }
            ],
        }
    ],
    "unmatched_documents": [
        {
            "id": "cand-A",
            "links": [
                {
                    "entity_name": "JPMorgan Chase",
                    "direction": "outgoing",
                    "settlement_date": "2025-06-15",
                }
            ],
        },
        {
            "id": "cand-B",
            "links": [
                {
                    "entity_name": "Goldman Sachs Group",
                    "direction": "outgoing",
                    "settlement_date": "2025-06-15",
                }
            ],
        },
        {
            "id": "cand-C",
            "links": [
                {
                    "entity_name": "JP Morgan",
                    "direction": "outgoing",
                    "settlement_date": "2025-06-15",
                }
            ],
        },
    ],
}

model = KKEdgeMatchingModel(config_dict=config_fuzzy, return_explainability=True)
results = model.match(contract_fuzzy)

r = results[0]
print(f"Matched: {r.matched}")
print(f"Best candidate: {r.best_candidate_document_id}")
print(f"Best score: {r.best_score:.4f}\n")

print("All candidates:")
print(f"  {'Candidate':<15s}  {'Entity Score':>12s}  {'Overall':>8s}")
print(f"  {'-'*15}  {'-'*12}  {'-'*8}")
for c in r.explanation.candidates:
    entity_detail = next(fd for fd in c.field_details if fd.field_name == "entity_name")
    print(f"  {c.candidate_document_id:<15s}  {entity_detail.score:>12.4f}  {c.overall_score:>8.4f}")

assert r.matched is True
assert r.best_candidate_document_id == "cand-A"
print("\nAssertions passed.")

Matched: True
Best candidate: cand-A
Best score: 0.9118

All candidates:
  Candidate        Entity Score   Overall
  ---------------  ------------  --------
  cand-A                 0.8235    0.9118
  cand-B                 0.4615    0.7308
  cand-C                 0.5517    0.7759

Assertions passed.


---

## Part 2 — Semantic Match (requires sentence-transformers)

Semantic matching captures meaning-based similarity that fuzzy matching cannot:
- "Amount Paid" vs "Money Received" (different words, related meaning)
- "Legal Entity" vs "Counterparty" (domain synonyms)

> **Install**: `pip install matching-engine[semantic]`
>
> If not installed, cells below will show a clear skip message.

In [6]:
try:
    import sentence_transformers  # noqa: F401
    SEMANTIC_AVAILABLE = True
    print(f"sentence-transformers {sentence_transformers.__version__} available")
except ImportError:
    SEMANTIC_AVAILABLE = False
    print("sentence-transformers NOT installed.")
    print("Semantic experiments will be skipped.")
    print("Install with: pip install matching-engine[semantic]")

sentence-transformers NOT installed.
Semantic experiments will be skipped.
Install with: pip install matching-engine[semantic]


### 2.1 — Semantic Pair Scoring

Compare individual pairs to see how semantic similarity captures meaning.

In [7]:
if not SEMANTIC_AVAILABLE:
    print("SKIPPED — sentence-transformers not installed")
else:
    from nerds_nlp.models.edge.matching.strategies import SemanticMatchStrategy

    semantic = SemanticMatchStrategy()

    pairs = [
        # Semantic synonyms (fuzzy would miss these)
        ("Amount Paid", "Money Received"),
        ("Legal Entity", "Counterparty"),
        ("Settlement Date", "Payment Date"),
        ("Forward Rate", "Exchange Rate"),
        ("Trade Confirmation", "Deal Confirmation"),
        # Identical
        ("Amount Paid", "Amount Paid"),
        # Unrelated
        ("Amount Paid", "Blue Whale"),
        ("Settlement Date", "Chocolate Cake"),
    ]

    print(f"{'Input':<25s}  {'Candidate':<25s}  {'Semantic':>8s}  {'Fuzzy':>6s}")
    print(f"{'-'*25}  {'-'*25}  {'-'*8}  {'-'*6}")

    fuzzy_cmp = FuzzyMatchStrategy()
    for a, b in pairs:
        sem_score = semantic.score_pair(a, b)
        fuz_score = fuzzy_cmp.score_pair(a, b)
        print(f"{a:<25s}  {b:<25s}  {sem_score:>8.4f}  {fuz_score:>6.4f}")

SKIPPED — sentence-transformers not installed


### 2.2 — One-to-Many Semantic Ranking

Rank candidate field labels by semantic similarity to an input label.

In [8]:
if not SEMANTIC_AVAILABLE:
    print("SKIPPED — sentence-transformers not installed")
else:
    input_label = "Amount Paid"
    candidate_labels = [
        "Amount Paid",
        "Money Received",
        "Payment Amount",
        "Total Cost",
        "Sum Transferred",
        "Settlement Date",
        "Blue Whale",
        "Chocolate Cake",
    ]

    scores = semantic.score_many(input_label, candidate_labels)
    ranked = sorted(zip(candidate_labels, scores), key=lambda x: x[1], reverse=True)

    print(f"Input: '{input_label}'\n")
    print(f"{'Rank':>4s}  {'Candidate':<25s}  {'Score':>6s}")
    print(f"{'-'*4}  {'-'*25}  {'-'*6}")
    for rank, (label, score) in enumerate(ranked, 1):
        print(f"{rank:>4d}  {label:<25s}  {score:>6.4f}")

SKIPPED — sentence-transformers not installed


### 2.3 — Side-by-Side: Fuzzy vs Semantic

Demonstrate cases where semantic matching outperforms fuzzy matching.

In [9]:
if not SEMANTIC_AVAILABLE:
    print("SKIPPED — sentence-transformers not installed")
else:
    test_cases = [
        # (input, candidate, description)
        ("Amount Paid", "Money Received", "Semantic synonym"),
        ("Legal Entity", "Counterparty", "Domain synonym"),
        ("JPMorgan Chase", "JPMorgan", "Substring match"),
        ("ABC Corporation", "ABC Corp", "Abbreviation"),
        ("Trade Confirmation", "Deal Confirmation", "Near synonym"),
        ("Forward Rate", "Exchange Rate", "Related concept"),
        ("Notional Amount", "Principal Value", "Financial synonym"),
        ("Amount Paid", "Blue Whale", "Unrelated"),
    ]

    fuzzy_cmp = FuzzyMatchStrategy()

    print(f"{'Input':<22s}  {'Candidate':<22s}  {'Fuzzy':>6s}  {'Semantic':>8s}  {'Winner':>10s}  {'Description'}")
    print(f"{'-'*22}  {'-'*22}  {'-'*6}  {'-'*8}  {'-'*10}  {'-'*20}")
    for a, b, desc in test_cases:
        fuz = fuzzy_cmp.score_pair(a, b)
        sem = semantic.score_pair(a, b)
        winner = "Semantic" if sem > fuz else ("Fuzzy" if fuz > sem else "Tie")
        print(f"{a:<22s}  {b:<22s}  {fuz:>6.4f}  {sem:>8.4f}  {winner:>10s}  {desc}")

SKIPPED — sentence-transformers not installed


### 2.4 — End-to-End Semantic Matching via KKEdgeMatchingModel

Use `semantic_match` as a field strategy in the full pipeline.

In [10]:
if not SEMANTIC_AVAILABLE:
    print("SKIPPED — sentence-transformers not installed")
else:
    config_semantic = {
        "matching": {
            "threshold": 0.50,
            "fields": [
                {
                    "name": "entity_name",
                    "weight": 0.4,
                    "strategy": "fuzzy_match",
                    "params": {"threshold": 0.4},
                },
                {
                    "name": "label",
                    "weight": 0.4,
                    "strategy": "semantic_match",
                },
                {
                    "name": "direction",
                    "weight": 0.2,
                    "strategy": "direction_inverse",
                },
            ],
        }
    }

    contract_semantic = {
        "input_documents": [
            {
                "id": "input-1",
                "links": [
                    {
                        "entity_name": "JPMorgan Chase",
                        "label": "Amount Paid",
                        "direction": "incoming",
                    }
                ],
            }
        ],
        "unmatched_documents": [
            {
                "id": "cand-A-semantic-hit",
                "links": [
                    {
                        "entity_name": "JPMorgan Chase & Co.",
                        "label": "Money Received",
                        "direction": "outgoing",
                    }
                ],
            },
            {
                "id": "cand-B-unrelated",
                "links": [
                    {
                        "entity_name": "Goldman Sachs",
                        "label": "Blue Whale",
                        "direction": "outgoing",
                    }
                ],
            },
            {
                "id": "cand-C-fuzzy-only",
                "links": [
                    {
                        "entity_name": "JPMorgan",
                        "label": "Settlement Date",
                        "direction": "outgoing",
                    }
                ],
            },
        ],
    }

    model = KKEdgeMatchingModel(config_dict=config_semantic, return_explainability=True)
    results = model.match(contract_semantic)

    r = results[0]
    print(f"Matched: {r.matched}")
    print(f"Best candidate: {r.best_candidate_document_id}")
    print(f"Best score: {r.best_score:.4f}\n")

    print(f"{'Candidate':<25s}  {'entity (fuzzy)':>14s}  {'label (semantic)':>16s}  {'direction':>10s}  {'overall':>8s}")
    print(f"{'-'*25}  {'-'*14}  {'-'*16}  {'-'*10}  {'-'*8}")
    for c in r.explanation.candidates:
        fd = {d.field_name: d for d in c.field_details}
        print(
            f"{c.candidate_document_id:<25s}  "
            f"{fd['entity_name'].score:>14.4f}  "
            f"{fd['label'].score:>16.4f}  "
            f"{fd['direction'].score:>10.4f}  "
            f"{c.overall_score:>8.4f}"
        )

    # cand-A should win: strong fuzzy on entity + strong semantic on label + direction match
    assert r.best_candidate_document_id == "cand-A-semantic-hit"
    print("\nAssertions passed.")

SKIPPED — sentence-transformers not installed


---

## Summary

| Scenario | Fuzzy handles it? | Semantic handles it? |
|---|---|---|
| "JPMorgan Chase" vs "JPMorgan" | Yes | Yes |
| "ABC Corp" vs "ABC Corporation" | Yes | Yes |
| "Amount Paid" vs "Money Received" | **No** | **Yes** |
| "Legal Entity" vs "Counterparty" | **No** | **Yes** |
| "Forward Rate" vs "Exchange Rate" | **No** | **Yes** |

**Recommendation**: Use `fuzzy_match` for entity names, identifiers, and codes.
Use `semantic_match` for descriptive labels and free-text fields where meaning matters.